# 08 · Factores asociados a la estancia: análisis bivariado

Asociación de cada factor con la estancia en urgencias (LOS > 12 h y mediana del LOS), por separado en atención en urgencias y hospitalizados, sin modelos.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().resolve()
while not (RAIZ / 'src' / 'config.py').exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import matplotlib.pyplot as plt
import pandas as pd

from src import bivariado as bv
from src import config
from src.graficos import activar_formato_tablas, aplicar_estilo, guardar_figura, titulo_hallazgo
from src.privacidad import suprimir_pequenos
from src.tablas import mostrar_tabla, tabla_medidas

aplicar_estilo()
activar_formato_tablas()
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 160)

## 1. Carga y población

Tabla de características del notebook 07. Se analizan los episodios de atención en urgencias y hospitalizados válidos para tiempos, fuera del periodo de calentamiento y con LOS; las variables de historia usan solo los episodios con su ventana completa.

In [ ]:
ruta = config.DATA_INTERIM / 'episodios_caracteristicas.parquet'
if not ruta.exists():
    tabla_medidas({'archivo': 'data/interim/episodios_caracteristicas.parquet', 'aviso': 'El archivo no existe.',
                   'qué hacer': 'Corre el notebook 07 para crear la tabla de características.'}, 'Tabla no disponible')
    raise SystemExit('Tabla no disponible: la celda se detiene.')

caracteristicas = pd.read_parquet(ruta)
episodios, flujo = bv.poblacion_bivariado(caracteristicas)
mostrar_tabla(suprimir_pequenos(flujo, list(flujo.columns), conservar_ceros=True),
              'Episodios por grupo después de cada filtro')

predictores = [c for c, rol in config.ROLES_CARACTERISTICAS.items() if rol == 'predictor']
sin_familia = [c for c in predictores if not any(c in variables for variables in bv.FAMILIAS.values())]
if sin_familia:
    tabla_medidas({'factores sin familia': ', '.join(sin_familia)}, 'Aviso: factores que no se analizan')
tablas = {}

## 2. Asociación de cada factor con la estancia

Para cada factor y grupo: n, % con LOS > 12 h con IC de Wilson y mediana del LOS con percentiles 25 y 75, por categoría o por decil (calculado dentro de cada grupo, con sus cortes en la tabla). Hora, día de la semana, edad, sexo y hospitalización previa solo tienen tabla: su gráfico está en el notebook 04.

### 2.1 Calendario

In [ ]:
tablas.update(bv.mostrar_familia(episodios, 'calendario', '08_calendario'))

### 2.2 Paciente

In [ ]:
tablas.update(bv.mostrar_familia(episodios, 'paciente', '08_paciente'))

### 2.3 Historia previa

Solo episodios con la ventana de historia de la variable completa (30 o 365 días).

In [ ]:
tablas.update(bv.mostrar_familia(episodios, 'historia previa', '08_historia_previa'))

### 2.4 Cola en urgencias

In [ ]:
tablas.update(bv.mostrar_familia(episodios, 'cola en urgencias', '08_cola_urgencias'))

### 2.5 Ocupación del piso

In [ ]:
tablas.update(bv.mostrar_familia(episodios, 'ocupación del piso', '08_ocupacion_piso'))

### 2.6 LOS reciente

In [ ]:
tablas.update(bv.mostrar_familia(episodios, 'LOS reciente', '08_los_reciente'))

## 3. Tamaño de efecto

Por variable y grupo: diferencia entre la categoría con mayor y menor % > 12 h (solo categorías con al menos 10 episodios), V de Cramér entre la categoría y LOS > 12 h y, para las continuas, correlación de Spearman con el LOS en horas. La tabla y el gráfico se ordenan por V de Cramér; el gráfico no incluye las versiones absolutas de la ocupación ni las versiones redundantes de la edad (edad en años y pediatría).

In [ ]:
efectos = pd.concat([bv.efecto_variable(episodios, variable, tabla) for variable, tabla in tablas.items()],
                    ignore_index=True)
familia_de = {v: familia for familia, variables in bv.FAMILIAS.items() for v in variables}
efectos.insert(2, 'familia', efectos['variable'].map(familia_de))
efectos = efectos.sort_values(['grupo', 'v_cramer'], ascending=[True, False])
mostrar_tabla(suprimir_pequenos(efectos.set_index(['grupo', 'variable']).round(3), ['n']),
              'Tamaño de efecto por variable y grupo')

In [ ]:
fig = bv.grafico_efecto(efectos)
titulo_hallazgo(fig, 'Tamaño de efecto de cada factor',
                f'V de Cramér entre las categorías del factor y LOS > {config.UMBRAL_OPERATIVO_H} h, por grupo')
guardar_figura(fig, '08_tamano_efecto')
plt.show()

## 4. Tendencia temporal

% con LOS > 12 h, mediana del LOS y número de llegadas por trimestre de llegada, para cada grupo.

In [ ]:
tendencia = bv.tendencia_trimestral(episodios)
mostrar_tabla(bv.tabla_segura_bivariado(tendencia), 'Estancia y llegadas por trimestre y grupo')

fig = bv.grafico_tendencia(tendencia)
titulo_hallazgo(fig, 'Estancia en urgencias por trimestre',
                f'% con LOS > {config.UMBRAL_OPERATIVO_H} h (IC 95 % de Wilson), mediana del LOS y llegadas por trimestre')
guardar_figura(fig, '08_tendencia_trimestral')
plt.show()

## Exportación

Tablas agregadas en `data/processed/`; los conteos menores a 10 se suprimen.

In [ ]:
config.DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
bivariado = bv.tabla_exportacion(tablas)
bivariado.to_csv(config.DATA_PROCESSED / 'factores_bivariado.csv', index=False)
efecto_csv = suprimir_pequenos(efectos.round(4), ['n'], columnas_derivadas=['diferencia_max_pp', 'v_cramer', 'spearman'])
efecto_csv.to_csv(config.DATA_PROCESSED / 'factores_efecto.csv', index=False)
mostrar_tabla(pd.DataFrame({'filas': {'factores_bivariado.csv': len(bivariado), 'factores_efecto.csv': len(efecto_csv)},
                            'columnas': {'factores_bivariado.csv': bivariado.shape[1],
                                         'factores_efecto.csv': efecto_csv.shape[1]}}).rename_axis('archivo'),
              'Archivos exportados en data/processed/')